# Neural Identifier Training with Particle Filters - Van der Pol Oscillator

In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
# ============================================================
# 1) True nonlinear system (Van der Pol Oscillator)
# ============================================================
def plant_dynamics(x, u, mu=1.0):
    """
    Continuous dynamics for Van der Pol oscillator: x = [x1, x2]. 
    Returns x_dot.
    
    The Van der Pol equations:
    dx1/dt = x2
    dx2/dt = μ(1 - x1²)x2 - x1
    """
    x1, x2 = x
    
    # Van der Pol equations
    x1_dot = x2
    x2_dot = mu * (1 - x1**2) * x2 - x1
    
    return np.array([x1_dot, x2_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

In [3]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est):
    """
    Features for a 2-state Van der Pol oscillator (no inputs):
    z = [S(x1), S(x2), S(x1)S(x2), S(x1)^2, S(x2)^2, S(x1)^3, x1, x2, 1]
    """
    s_x1 = sigmoidal(x_est[0])  # x1 (position)
    s_x2 = sigmoidal(x_est[1])  # x2 (velocity)
    
    return np.array([
        s_x1, s_x2,                           # Sigmoid terms
        s_x1*s_x2,                            # Cross term
        s_x1**2, s_x2**2,                     # Quadratic sigmoid terms
        s_x1**3,                              # Cubic term (important for Van der Pol)
        x_est[0], x_est[1],                   # Linear terms (direct states)
        1.0                                    # Bias
    ])


def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [4]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x1 (position - measured output for Van der Pol oscillator)

        z_i = construct_z_vector(x_state_for_z)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

In [5]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x (measured output for Lorenz system)
        z = construct_z_vector(x_state_for_z)  # (num_features,)

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]


In [6]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        # Calculate lambda and weights for sigma points
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Weights for mean and covariance computation
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        """Generate sigma points for UKF."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x (measured output for Lorenz system)

        z_i = construct_z_vector(x_state_for_z)  # shape (num_features,)

        for i in range(self.num_neurons):
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

In [7]:
# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian'  # 'laplacian' | 'uniform' | 'gaussian'
    process_noise_std = 0.01

    # --- True system init ---
    x_true = np.zeros((n_steps, 2))
    x_true[0] = [2.0, 0.0]  # Initial conditions for Van der Pol oscillator [position, velocity]
    u = 0.0

    # --- RHONN config ---
    num_neurons = 2  # Two states for Van der Pol oscillator
    num_features = 9  # Updated feature vector size for 2 states
    num_weights_per_neuron = num_features

    # --- Common initial weights for fair comparison ---
    np.random.seed(7517)  # (optional) reproducibility of initial weights
    common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i}: {w}")

    # # --- EKF --- (Tuned parameters for Van der Pol oscillator)
    # ekf_trainer = EKF_RHONN_Trainer(
    #     num_neurons, num_weights_per_neuron,
    #     initial_weights=common_initial_weights,
    #     Q_init=1e-3, R_init=1e-2, P_init=1.0, eta=0.5
    # )
    # x_hat_ekf = np.zeros((n_steps, 2))
    # x_hat_ekf[0] = x_true[0]

    # --- UKF ---
    ukf_trainer = UKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1e-3, R_init=1e-2, P_init=1.0, eta=0.8,
        alpha=1e-3, beta=2.0  # UKF-specific parameters
    )
    x_hat_ukf = np.zeros((n_steps, 2))
    x_hat_ukf[0] = x_true[0]

    # --- PF ---
    n_particles = 200
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=0.3, R_std=np.sqrt(0.01), ess_threshold=n_particles / 2  # ESS < N/2
    )

    # Force identical particle initialization if desired:
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    x_hat_pf = np.zeros((n_steps, 2))
    x_hat_pf[0] = x_true[0]

    print("Starting simulation...")
    for k in range(n_steps - 1):
        # ---- 1) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u, dt, process_noise_type, process_noise_std)

        # # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        # ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ekf[k])

        # x_state_for_z_ekf = np.copy(x_hat_ekf[k])
        # x_state_for_z_ekf[0] = x_true[k][0]  # series-parallel uses measured x1 at k
        # x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0])  # x1
        # x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1])  # x2

        # ---- 2b) UKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ukf[k])

        x_state_for_z_ukf = np.copy(x_hat_ukf[k])
        x_state_for_z_ukf[0] = x_true[k][0]  # series-parallel uses measured x1 at k
        x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0])  # x1
        x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1])  # x2

        # ---- 3) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k])

        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0]  # series-parallel uses measured x1 at k
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0])   # x1
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1])   # x2

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")


Common Initial Weights:
  Neuron 0: [ 0.09193582 -0.35502595  0.15823179 -0.27779915  0.49295881 -0.33445916
 -0.28976594  0.38618909 -0.38816842]
  Neuron 1: [-0.18599621 -0.34524321 -0.03517789  0.37400109 -0.04928347 -0.41190637
  0.18537181  0.02549788  0.07931941]
Starting simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 60.0%
Simulation progress: 70.0%
Simulation progress: 80.0%
Simulation progress: 90.0%
Simulation finished.


In [8]:
# ============================================================
# 6) Resultados y gráficas para Oscilador de Van der Pol
# ============================================================

# Configuración de formato para tesis
thesis_config = {
    'font_family': 'Computer Modern, serif',
    'font_size': 14,
    'title_font_size': 16,
    'legend_font_size': 12,
    'line_width_true': 2.5,
    'line_width_est': 2.0,
    'plot_width': 1000,
    'plot_height': 500,
    'grid_color': 'rgba(200, 200, 200, 0.3)',
    'grid_width': 0.5
}

# Cálculo de MSE
# mse_x1_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
# mse_x2_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
mse_x1_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
mse_x2_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
mse_x1_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
mse_x2_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)

# mse_total_ekf = mse_x1_ekf + mse_x2_ekf
mse_total_ukf = mse_x1_ukf + mse_x2_ukf
mse_total_pf = mse_x1_pf + mse_x2_pf

print("="*70)
print(f"🏆 MEJOR FILTRO: ", end="")
mse_dict = {'UKF': mse_total_ukf, 'PF': mse_total_pf}  # EKF commented out
best_filter = min(mse_dict, key=mse_dict.get)
print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.6f})")
print("="*70)

# print(f"\nPesos Finales EKF-RHONN:")
# for i in range(2):
#     print(f"  Neurona {i+1} (x{i+1}): {ekf_trainer.weights[i]}")

print(f"\nPesos Finales UKF-RHONN:")
for i in range(2):
    print(f"  Neurona {i+1} (x{i+1}): {ukf_trainer.weights[i]}")

print(f"\nEstimación de Pesos PF-RHONN:")
pf_estimates = pf_trainer.get_estimate()
for i in range(2):
    print(f"  Neurona {i+1} (x{i+1}): {pf_estimates[i]}")

print("\n--- Comparación de Desempeño (MSE) - Oscilador de Van der Pol ---")
# print(f"EKF MSE x₁ (posición):  {mse_x1_ekf:.6f}")
# print(f"EKF MSE x₂ (velocidad): {mse_x2_ekf:.6f}")
print(f"UKF MSE x₁ (posición):  {mse_x1_ukf:.6f}")
print(f"UKF MSE x₂ (velocidad): {mse_x2_ukf:.6f}")
print(f"PF  MSE x₁ (posición):  {mse_x1_pf:.6f}")
print(f"PF  MSE x₂ (velocidad): {mse_x2_pf:.6f}")

# Gráficas individuales por estado
states_info = [
    {'idx': 0, 'var': 'x₁', 'desc': 'Posición', 'y_label': 'Posición x₁'},
    {'idx': 1, 'var': 'x₂', 'desc': 'Velocidad', 'y_label': 'Velocidad x₂'}
]

for state_info in states_info:
    i = state_info['idx']
    
    fig = go.Figure()
    
    # Estado real (línea negra gruesa)
    fig.add_trace(go.Scatter(
        x=t_history, y=x_true[:, i],
        mode='lines',
        name='Estado Real',
        line=dict(color='#000000', width=thesis_config['line_width_true']),
        showlegend=True
    ))
    
    # # Estimación EKF
    # fig.add_trace(go.Scatter(
    #     x=t_history, y=x_hat_ekf[:, i],
    #     mode='lines',
    #     name='EKF-RHONN',
    #     line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
    #     showlegend=True
    # ))
    
    # Estimación UKF
    fig.add_trace(go.Scatter(
        x=t_history, y=x_hat_ukf[:, i],
        mode='lines',
        name='UKF-RHONN',
        line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
        showlegend=True
    ))
    
    # Estimación PF
    fig.add_trace(go.Scatter(
        x=t_history, y=x_hat_pf[:, i],
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
        showlegend=True
    ))
    
    fig.update_layout(
        title={
            'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Oscilador de Van der Pol',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title=state_info['y_label'],
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig.show()

# Gráfica de errores combinada
# error_x1_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
# error_x2_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_x1_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_x2_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
error_x1_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_x2_pf = x_true[:, 1] - x_hat_pf[:, 1]

fig_err = go.Figure()

# # Errores x1
# fig_err.add_trace(go.Scatter(
#     x=t_history, y=error_x1_ekf,
#     mode='lines',
#     name=f'EKF Error x₁ (MSE={mse_x1_ekf:.2e})',
#     line=dict(color='#1f77b4', width=1.5),
#     opacity=0.8
# ))

fig_err.add_trace(go.Scatter(
    x=t_history, y=error_x1_ukf,
    mode='lines',
    name=f'UKF Error x₁ (MSE={mse_x1_ukf:.2e})',
    line=dict(color='#2ca02c', width=1.5),
    opacity=0.8
))

fig_err.add_trace(go.Scatter(
    x=t_history, y=error_x1_pf,
    mode='lines',
    name=f'PF Error x₁ (MSE={mse_x1_pf:.2e})',
    line=dict(color='#d62728', width=1.5),
    opacity=0.8
))

# # Errores x2
# fig_err.add_trace(go.Scatter(
#     x=t_history, y=error_x2_ekf,
#     mode='lines',
#     name=f'EKF Error x₂ (MSE={mse_x2_ekf:.2e})',
#     line=dict(color='#1f77b4', width=1.5, dash='dot'),
#     opacity=0.8
# ))

fig_err.add_trace(go.Scatter(
    x=t_history, y=error_x2_ukf,
    mode='lines',
    name=f'UKF Error x₂ (MSE={mse_x2_ukf:.2e})',
    line=dict(color='#2ca02c', width=1.5, dash='dot'),
    opacity=0.8
))

fig_err.add_trace(go.Scatter(
    x=t_history, y=error_x2_pf,
    mode='lines',
    name=f'PF Error x₂ (MSE={mse_x2_pf:.2e})',
    line=dict(color='#d62728', width=1.5, dash='dot'),
    opacity=0.8
))

fig_err.update_layout(
    title={
        'text': 'Errores de Estimación - Oscilador de Van der Pol',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo (s)',
    yaxis_title='Error de Estimación',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_err.show()

# Espacio de fase (Ciclo Límite)
fig_phase = go.Figure()

fig_phase.add_trace(go.Scatter(
    x=x_true[:, 0], y=x_true[:, 1],
    mode='lines',
    name='Ciclo Límite Real',
    line=dict(color='#000000', width=3)
))

# fig_phase.add_trace(go.Scatter(
#     x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1],
#     mode='lines',
#     name='Estimación EKF-RHONN',
#     line=dict(color='#1f77b4', width=2, dash='dash')
# ))

fig_phase.add_trace(go.Scatter(
    x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1],
    mode='lines',
    name='Estimación UKF-RHONN',
    line=dict(color='#2ca02c', width=2, dash='dot')
))

fig_phase.add_trace(go.Scatter(
    x=x_hat_pf[:, 0], y=x_hat_pf[:, 1],
    mode='lines',
    name='Estimación PF-RHONN',
    line=dict(color='#d62728', width=2, dash='dashdot')
))

fig_phase.update_layout(
    title={
        'text': 'Espacio de Fases - Oscilador de Van der Pol',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Posición x₁',
    yaxis_title='Velocidad x₂',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_width'],  # Aspecto cuadrado
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_phase.show()

# Gráfica de barras comparando MSE
fig_mse = go.Figure()

filters = ['UKF-RHONN', 'PF-RHONN']  # EKF commented out

fig_mse.add_trace(go.Bar(
    name='Posición x₁',
    x=filters,
    y=[mse_x1_ukf, mse_x1_pf],
    marker_color='#636EFA',
    text=[f'{mse_x1_ukf:.2e}', f'{mse_x1_pf:.2e}'],
    textposition='outside'
))

fig_mse.add_trace(go.Bar(
    name='Velocidad x₂',
    x=filters,
    y=[mse_x2_ukf, mse_x2_pf],
    marker_color='#EF553B',
    text=[f'{mse_x2_ukf:.2e}', f'{mse_x2_pf:.2e}'],
    textposition='outside'
))

fig_mse.update_layout(
    title={
        'text': 'Comparación de Error Cuadrático Medio (MSE)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tipo de Filtro',
    yaxis_title='Error Cuadrático Medio (MSE)',
    yaxis=dict(
        type='log',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    xaxis=dict(
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    barmode='group',
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_mse.show()


🏆 MEJOR FILTRO: UKF (MSE total: 0.000784)

Pesos Finales UKF-RHONN:
  Neurona 1 (x1): [ 0.6550011  -1.30858208  0.52001847  0.24279605  0.42579649  0.06667199
  0.95797664 -0.10121039 -0.06439721]
  Neurona 2 (x2): [-0.47432187  1.60583708 -0.03280318  0.20584725  0.32796577 -0.50638813
  0.09323448  1.17139565  0.08985457]

Estimación de Pesos PF-RHONN:
  Neurona 1 (x1): [-6.11623774 -5.19650349  6.05064825 10.97228937  6.86996467 -0.82124821
 -1.72860405  3.49021562  0.07089261]
  Neurona 2 (x2): [-14.21648522  12.79481573 -14.58755271  15.24649537  -8.48454013
   8.41607307  -1.07621145   1.7926467   -1.78393753]

--- Comparación de Desempeño (MSE) - Oscilador de Van der Pol ---
UKF MSE x₁ (posición):  0.000560
UKF MSE x₂ (velocidad): 0.000224
PF  MSE x₁ (posición):  0.000770
PF  MSE x₂ (velocidad): 0.000155


# Double Pendulum Identification with RHONN

## Overview
Neural identification using three training algorithms:
- **EKF-RHONN**: Extended Kalman Filter
- **UKF-RHONN**: Unscented Kalman Filter  
- **PF-RHONN**: Particle Filter (neuron-specific parameters)

## Features
- RK4 discretization
- Non-Gaussian noise models
- NaN prevention strategies
- Comprehensive error metrics

## Execution
Run cells in order: Definitions → Simulation → Visualizations

In [ ]:
# Neural Identifier Training - Double Pendulum
# Methods: EKF, UKF, Particle Filter
# Con características específicas por neurona

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1) True nonlinear system (Double Pendulum)
# ============================================================
def plant_dynamics(state, u):
    """
    Dynamics of a double pendulum with damping and external torques.
    state = [θ1, θ2, ω1, ω2] 
        θ1, θ2: angles (rad)
        ω1, ω2: angular velocities (rad/s)
    u = [τ1, τ2]: external torques on joints (N⋅m)
    """
    # Physical Parameters
    m1 = 1.0     # Mass of first pendulum (kg)
    m2 = 1.0     # Mass of second pendulum (kg)
    l1 = 1.0     # Length of first pendulum (m)
    l2 = 1.0     # Length of second pendulum (m)
    g = 9.81     # Gravity (m/s²)
    b1 = 0.1     # Damping coefficient for joint 1 (N⋅m⋅s)
    b2 = 0.1     # Damping coefficient for joint 2 (N⋅m⋅s)
    
    θ1, θ2, ω1, ω2 = state
    τ1, τ2 = u
    
    # Precompute trigonometric terms
    c12 = np.cos(θ1 - θ2)  # cos(θ1 - θ2)
    s12 = np.sin(θ1 - θ2)  # sin(θ1 - θ2)
    s1 = np.sin(θ1)         # sin(θ1)
    s2 = np.sin(θ2)         # sin(θ2)
    
    # Mass matrix elements
    M11 = (m1 + m2) * l1**2
    M12 = m2 * l1 * l2 * c12
    M21 = M12
    M22 = m2 * l2**2
    
    # Coriolis and centrifugal terms
    h = m2 * l1 * l2 * ω2**2 * s12
    C1 = h - (m1 + m2) * g * l1 * s1 - b1 * ω1 + τ1
    
    C2 = -m2 * l1 * l2 * ω1**2 * s12 - m2 * g * l2 * s2 - b2 * ω2 + τ2
    
    # Solve for accelerations using the mass matrix
    # [M11  M12] [α1]   [C1]
    # [M21  M22] [α2] = [C2]
    det_M = M11 * M22 - M12 * M21
    
    if abs(det_M) < 1e-10:
        det_M = 1e-10  # Prevent division by zero
    
    α1 = (M22 * C1 - M12 * C2) / det_M
    α2 = (-M21 * C1 + M11 * C2) / det_M
    
    # State derivatives
    θ1_dot = ω1
    θ2_dot = ω2
    ω1_dot = α1
    ω2_dot = α2
    
    return np.array([θ1_dot, θ2_dot, ω1_dot, ω2_dot])

def plant(x_k, u_k, dt=0.01):
    """
    RK4 integration step for better accuracy.
    """
    # RK4 para mayor precisión
    k1 = plant_dynamics(x_k, u_k)
    k2 = plant_dynamics(x_k + 0.5*dt*k1, u_k)
    k3 = plant_dynamics(x_k + 0.5*dt*k2, u_k)
    k4 = plant_dynamics(x_k + dt*k3, u_k)
    
    x_kp1 = x_k + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)
    
    return x_kp1

# ============================================================
# 2) RHONN structure - CARACTERÍSTICAS POR NEURONA
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input, neuron_index):
    """
    Features for Double Pendulum - ESPECÍFICAS PARA CADA NEURONA.
    
    x_est = [θ1, θ2, ω1, ω2]
    u_input = [τ1, τ2]
    neuron_index: índice de la neurona (0=θ1, 1=θ2, 2=ω1, 3=ω2)
    """
    θ1, θ2, ω1, ω2 = x_est
    τ1, τ2 = u_input
    
    # Términos básicos sigmoidales
    s_θ1 = sigmoidal(θ1)
    s_θ2 = sigmoidal(θ2)
    s_ω1 = sigmoidal(ω1)
    s_ω2 = sigmoidal(ω2)
    
    # Términos trigonométricos (importantes para la dinámica del péndulo)
    # cos_θ1 = np.cos(θ1)
    # sin_θ1 = np.sin(θ1)
    # cos_θ2 = np.cos(θ2)
    # sin_θ2 = np.sin(θ2)
    # cos_θ12 = np.cos(θ1 - θ2)
    # sin_θ12 = np.sin(θ1 - θ2)
    
    # Comandos escalados
    s_τ1 = sigmoidal(τ1)
    s_τ2 = sigmoidal(τ2)
    
    # ========== CARACTERÍSTICAS ESPECÍFICAS POR NEURONA ==========
    
    if neuron_index == 0:  # Neurona para θ1 (ángulo del primer péndulo)
        # dθ1/dt = ω1
        return np.array([
            # s_ω1,                      # Velocidad angular (término principal)
            s_ω1**3,                   # Término cuadrático
            s_θ1**2,                      # Posición angular
            s_θ1 * s_ω1,              # Acoplamiento posición-velocidad
            # s_ω2,                      # Acoplamiento con segundo péndulo
            # cos_θ12,                   # Término de acoplamiento geométrico
        ])
    
    elif neuron_index == 1:  # Neurona para θ2 (ángulo del segundo péndulo)
        # dθ2/dt = ω2
        return np.array([
            s_ω2,                      # Velocidad angular (término principal)
            s_ω2**2,                   # Término cuadrático
            # s_θ2,                      # Posición angular
            s_θ2 * s_ω2,              # Acoplamiento posición-velocidad
            s_ω1,                      # Acoplamiento con primer péndulo
            # cos_θ12,                   # Término de acoplamiento geométrico
        ])
    
    elif neuron_index == 2:  # Neurona para ω1 (velocidad angular del primer péndulo)
        # dω1/dt = función compleja de θ1, θ2, ω1, ω2, τ1
        return np.array([
            s_ω1,                      # Estado actual
            s_ω1**2,                   # Término cuadrático (fricción)
            # sin_θ1,                    # Término gravitacional
            # sin_θ12,                   # Acoplamiento con segundo péndulo
            # s_ω2 * sin_θ12,           # Término de Coriolis
            s_τ1,                      # Torque externo
            # cos_θ12,                   # Acoplamiento geométrico
            s_ω1 * s_ω2,              # Interacción velocidades
        ])
    
    elif neuron_index == 3:  # Neurona para ω2 (velocidad angular del segundo péndulo)
        # dω2/dt = función compleja de θ1, θ2, ω1, ω2, τ2
        return np.array([
            s_ω2,                      # Estado actual
            s_ω2**2,                   # Término cuadrático (fricción)
            # sin_θ2,                    # Término gravitacional
            # sin_θ12,                   # Acoplamiento con primer péndulo
            # s_ω1 * sin_θ12,           # Término de Coriolis
            s_τ2,                      # Torque externo
            # cos_θ12,                   # Acoplamiento geométrico
            s_ω1 * s_ω2,              # Interacción velocidades
        ])
    
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# Función auxiliar para obtener el tamaño de características de cada neurona
def get_z_size(neuron_index):
    """Retorna el número de características para una neurona dada."""
    if neuron_index == 0:  # θ1
        return 3
    elif neuron_index == 1:  # θ2
        return 4
    elif neuron_index == 2:  # ω1
        return 4
    elif neuron_index == 3:  # ω2
        return 4
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# ============================================================
# 3) Trainers (EKF, UKF, PF) - ADAPTADOS PARA MÚLTIPLES TAMAÑOS
# ============================================================

class Generic_RHONN_Trainer:
    """ Base class to handle the loop logic easily """
    def get_prediction(self, weights, x_k, u_k, neuron_idx):
        z = construct_z_vector(x_k, u_k, neuron_idx)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    '''
    eta = 0.9780
   P0 = 3.9958
   Q = 1.24e-03
   R = 2.46e-04
    '''
    def __init__(self, n_neurons, eta=1.0, P0=1.0, Q=1e-3, R=1e-5):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i))*P0 for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*Q for i in range(n_neurons)]
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            # Construir vector de características H_i (ecuación 8)
            z = construct_z_vector(x_k, u_k, i)
            H_i = z.reshape(-1, 1)
            
            # Error de identificación e_i(k) (ecuación 7)
            # e_i(k) = x_i(k) - X̂_i(k)
            x_hat_i = np.dot(self.weights[i], z)
            e_i = x_kp1[i] - x_hat_i
            
            # Ganancia de Kalman K_i(k) (ecuación 6)
            # K_i(k) = P_i(k) H_i(k) [R_i(k) + H_i(k) P_i(k) H_i(k)]^{-1}
            S = self.R + (H_i.T @ self.P[i] @ H_i)[0, 0]
            K_i = (self.P[i] @ H_i).flatten() / S
            
            # Actualización de pesos ω_i(k+1) (ecuación superior)
            # ω_i(k+1) = ω_i(k) + η_i K_i(k) e_i(k)
            self.weights[i] = self.weights[i] + self.eta * K_i * e_i
            
            # Actualización de covarianza P_i(k+1) (ecuación 6, tercera línea)
            # P_i(k+1) = P_i(k) - K_i(k) H_i(k) P_i(k) + Q_i(k)
            self.P[i] = self.P[i] - np.outer(K_i, H_i.flatten()) @ self.P[i] + self.Q_matrices[i]

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, alpha=1e-2, Q_std=1e-3, R_std=1e-5):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i)) for i in range(n_neurons)]
        self.Q = Q_std
        self.Q_matrices = [np.eye(get_z_size(i))*Q_std for i in range(n_neurons)]
        self.R = R_std
        self.eta = eta
        self.alpha = alpha

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n = get_z_size(i)
            
            # Sigma params
            lambda_ = self.alpha**2 * n - n
            Wm = np.full(2*n+1, 1/(2*(n+lambda_)))
            Wc = np.copy(Wm)
            Wm[0] = lambda_/(n+lambda_)
            Wc[0] = Wm[0] + (3 - self.alpha**2)
            
            # Generate Sigmas
            try:
                L = np.linalg.cholesky((n + lambda_) * self.P[i])
            except:
                L = np.eye(n) * 0.1
                
            sigmas = np.zeros((2*n+1, n))
            sigmas[0] = self.weights[i]
            for k in range(n):
                sigmas[k+1] = self.weights[i] + L[:,k]
                sigmas[n+k+1] = self.weights[i] - L[:,k]
            
            # Transform
            Y_sigmas = np.dot(sigmas, z)
            y_mean = np.sum(Wm * Y_sigmas)
            
            # Covariances
            Py = np.sum(Wc * (Y_sigmas - y_mean)**2) + self.R
            Pxy = np.zeros(n)
            for k in range(2*n+1):
                Pxy += Wc[k] * (sigmas[k] - self.weights[i]) * (Y_sigmas[k] - y_mean)
                
            # Update
            K = Pxy / Py
            err = x_kp1[i] - y_mean
            self.weights[i] += self.eta * K * err
            self.P[i] -= np.outer(K, K) * Py
            
            # Regularize P
            self.P[i] += np.eye(n)*1e-6

class PF_Trainer(Generic_RHONN_Trainer):
    """
    Robust Particle Filter Trainer with NaN prevention strategies.
    Now supports neuron-specific Q and R values.
    """
    def __init__(self, n_neurons, n_particles=1000, Q_std=None, R_std=None):
        self.n_neurons = n_neurons
        self.n_particles = n_particles
        
        # Initialize particles with controlled variance
        self.particles = [np.random.randn(n_particles, get_z_size(i)) * 0.1 
                         for i in range(n_neurons)]
        
        # Initialize weights uniformly
        self.weights_pf = [np.ones(n_particles) / n_particles for _ in range(n_neurons)]
        
        # Tuning parameters - neuron-specific or default
        if Q_std is None:
            # Default: same for all neurons
            self.Q_std = [0.05] * n_neurons
        elif isinstance(Q_std, (list, np.ndarray)):
            # Use provided neuron-specific values
            assert len(Q_std) == n_neurons, f"Q_std must have length {n_neurons}"
            self.Q_std = list(Q_std)
        else:
            # Single value for all neurons
            self.Q_std = [Q_std] * n_neurons
        
        if R_std is None:
            # Default: same for all neurons
            self.R_std = [0.05] * n_neurons
        elif isinstance(R_std, (list, np.ndarray)):
            # Use provided neuron-specific values
            assert len(R_std) == n_neurons, f"R_std must have length {n_neurons}"
            self.R_std = list(R_std)
        else:
            # Single value for all neurons
            self.R_std = [R_std] * n_neurons
        
        self.regularization_std = 0.001  # Jitter after resampling
        
        # NaN prevention parameters
        self.min_weight = 1e-300    # Minimum weight to prevent underflow
        self.max_log_likelihood = 1000.0  # Clip extreme likelihoods
        self.resample_threshold = 0.3   # Resample when Neff < threshold * N
        
    def _normalize_weights(self, weights):
        """
        Safely normalize weights with NaN and underflow protection.
        """
        # Check for NaN or inf
        if np.any(~np.isfinite(weights)):
            print("⚠️  Warning: Non-finite weights detected, resetting to uniform")
            return np.ones_like(weights) / len(weights)
        
        # Ensure positive weights
        weights = np.maximum(weights, self.min_weight)
        
        # Normalize
        weight_sum = np.sum(weights)
        if weight_sum < self.min_weight or not np.isfinite(weight_sum):
            print("⚠️  Warning: Invalid weight sum, resetting to uniform")
            return np.ones_like(weights) / len(weights)
        
        return weights / weight_sum
    
    def _compute_log_likelihood(self, errors, neuron_idx):
        """
        Compute log-likelihood with numerical stability.
        Uses neuron-specific R_std.
        """
        # Clip extreme errors
        errors_clipped = np.clip(errors, -1000, 1000)
        
        # Gaussian log-likelihood: -0.5 * (err/sigma)^2
        # sigma = self.R_std[neuron_idx]
        # normalized_errors_sq = (errors_clipped / sigma) ** 2
        # log_likelihood = -0.5 * normalized_errors_sq
        
        # Student-t log-likelihood with df=3 (commented out)
        df = 3.0  # degrees of freedom (heavy tails)
        sigma = self.R_std[neuron_idx]
        normalized_errors_sq = (errors_clipped / sigma) ** 2
        log_likelihood = -((df + 1) / 2.0) * np.log(1.0 + normalized_errors_sq / df)
        
        return log_likelihood
    
    def _resample_particles(self, neuron_idx):
        """
        Systematic resampling with regularization.
        """
        weights = self.weights_pf[neuron_idx]
        particles = self.particles[neuron_idx]
        n_weights = get_z_size(neuron_idx)
        
        # Normalize weights
        weights = self._normalize_weights(weights)
        
        # Systematic resampling
        positions = (np.arange(self.n_particles) + np.random.random()) / self.n_particles
        cumulative_sum = np.cumsum(weights)
        
        indices = np.searchsorted(cumulative_sum, positions)
        indices = np.clip(indices, 0, self.n_particles - 1)
        
        # Resample particles
        self.particles[neuron_idx] = particles[indices].copy()
        
        # Add regularization jitter (preserve particle diversity)
        jitter = np.random.randn(self.n_particles, n_weights) * self.regularization_std
        self.particles[neuron_idx] += jitter
        
        # Reset weights to uniform
        self.weights_pf[neuron_idx] = np.ones(self.n_particles) / self.n_particles
    
    def update(self, x_kp1, x_k, u_k):
        """
        Update step with NaN prevention and neuron-specific Q/R.
        """
        for i in range(self.n_neurons):
            try:
                # Get feature vector
                z = construct_z_vector(x_k, u_k, i)
                n_weights = get_z_size(i)
                
                # Check for NaN in input
                if not np.all(np.isfinite(z)):
                    print(f"⚠️  Warning: Non-finite feature vector for neuron {i}, skipping update")
                    continue
                
                # 1. PREDICTION: Add process noise (drift) - neuron-specific Q_std
                drift = np.random.randn(self.n_particles, n_weights) * self.Q_std[i]
                self.particles[i] += drift
                
                # Clip particles to prevent extreme values
                self.particles[i] = np.clip(self.particles[i], -1000, 1000)
                
                # 2. UPDATE: Compute likelihoods
                preds = self.particles[i] @ z
                
                # Check for NaN in predictions
                if not np.all(np.isfinite(preds)):
                    print(f"⚠️  Warning: Non-finite predictions for neuron {i}, resetting particles")
                    self.particles[i] = np.random.randn(self.n_particles, n_weights) * 0.1
                    continue
                
                # Compute errors
                errors = x_kp1[i] - preds
                
                # Compute log-likelihood (numerically stable) - neuron-specific R_std
                log_likelihood = self._compute_log_likelihood(errors, i)
                
                # Shift for numerical stability before exp
                log_likelihood -= np.max(log_likelihood)
                
                # Update weights
                self.weights_pf[i] *= np.exp(log_likelihood)
                
                # Normalize weights
                self.weights_pf[i] = self._normalize_weights(self.weights_pf[i])
                
                # 3. RESAMPLING: Check effective sample size
                weight_sq_sum = np.sum(self.weights_pf[i] ** 2)
                if weight_sq_sum > 0:
                    eff_N = 1.0 / weight_sq_sum
                else:
                    eff_N = 0
                
                # Resample if effective sample size is too low
                if eff_N < self.resample_threshold * self.n_particles:
                    self._resample_particles(i)
                    
            except Exception as e:
                print(f"⚠️  Error in PF update for neuron {i}: {e}")
                # Reset to safe state
                self.particles[i] = np.random.randn(self.n_particles, get_z_size(i)) * 0.1
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
    
    def get_estimates(self):
        """
        Get weighted average of particles (with NaN protection).
        """
        estimates = []
        for i in range(self.n_neurons):
            # Normalize weights
            weights = self._normalize_weights(self.weights_pf[i])
            
            # Compute weighted average
            estimate = np.average(self.particles[i], axis=0, weights=weights)
            
            # Check for NaN
            if not np.all(np.isfinite(estimate)):
                print(f"⚠️  Warning: Non-finite estimate for neuron {i}, using median")
                estimate = np.median(self.particles[i], axis=0)
            
            estimates.append(estimate)
        
        return estimates

# ============================================================
# 4) Error Metrics Functions
# ============================================================
def calculate_error_metrics(y_true, y_pred, metric_name="State"):
    """
    Calculate comprehensive error metrics.
    
    Args:
        y_true: True values (n_samples, n_states)
        y_pred: Predicted values (n_samples, n_states)
        metric_name: Name for reporting
    
    Returns:
        Dictionary with error metrics
    """
    errors = y_true - y_pred
    
    # Mean Absolute Error (MAE)
    mae = np.mean(np.abs(errors), axis=0)
    mae_total = np.mean(mae)
    
    # Root Mean Square Error (RMSE)
    rmse = np.sqrt(np.mean(errors**2, axis=0))
    rmse_total = np.sqrt(np.mean(rmse**2))
    
    # Normalized RMSE (NRMSE) - normalized by range
    ranges = np.max(y_true, axis=0) - np.min(y_true, axis=0)
    ranges[ranges < 1e-10] = 1.0  # Avoid division by zero
    nrmse = rmse / ranges
    nrmse_total = np.mean(nrmse)
    
    # Maximum Absolute Error
    max_error = np.max(np.abs(errors), axis=0)
    
    # Mean Squared Error (MSE)
    mse = np.mean(errors**2, axis=0)
    mse_total = np.mean(mse)
    
    # R² Score (coefficient of determination)
    ss_res = np.sum(errors**2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0))**2, axis=0)
    r2 = 1 - (ss_res / (ss_tot + 1e-10))
    r2_total = np.mean(r2)
    
    return {
        'MAE': mae,
        'MAE_total': mae_total,
        'RMSE': rmse,
        'RMSE_total': rmse_total,
        'NRMSE': nrmse,
        'NRMSE_total': nrmse_total,
        'MSE': mse,
        'MSE_total': mse_total,
        'Max_Error': max_error,
        'R2': r2,
        'R2_total': r2_total
    }

In [ ]:
import time  # Añadir al inicio del archivo

# ============================================================
# 5) Simulation Main Loop - DOUBLE PENDULUM
# ============================================================
if __name__ == "__main__":
    # ============================================================
    # Main Simulation - PARALLEL CONFIGURATION
    # ============================================================
    n_steps = 1500
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    
    n_states = 4  # [θ1, θ2, ω1, ω2]
    
    # Gaussian noise parameters
    # Process noise (affects true state evolution)
    process_noise_std = [0.001, 0.001, 0.002, 0.002]  # [θ1, θ2, ω1, ω2]
    
    # Measurement noise parameters
    angle_noise_std = 0.01        # 0.01 rad (~0.57°) angle error
    omega_noise_std = 0.02        # 0.02 rad/s angular velocity error
    
    # ============================================================
    # GENERATE INITIAL WEIGHTS (UNIFORM DISTRIBUTION) - SHARED BY ALL FILTERS
    # ============================================================
    np.random.seed(7517)  # For reproducibility
    initial_weights = []
    for i in range(n_states):
        # Uniform distribution U(-1.0, 1.0)
        w_init = np.random.uniform(-1.0, 1.0, size=get_z_size(i))
        initial_weights.append(w_init.copy())
    
    print("Initial RHONN weights (uniform distribution):")
    for i, w in enumerate(initial_weights):
        print(f"  Neuron {i}: shape={w.shape}, min={w.min():.4f}, max={w.max():.4f}, mean={w.mean():.4f}")
    
    # ============================================================
    # PF-RHONN Neuron-Specific Q and R values
    # ============================================================
    pf_Q_std = [
        0.5,   # Neuron 0 (θ1): angle state
        0.5,   # Neuron 1 (θ2): angle state
        0.2,   # Neuron 2 (ω1): velocity state (slightly higher dynamics)
        0.2    # Neuron 3 (ω2): velocity state (slightly higher dynamics)
    ]
    
    pf_R_std = [
        1.0e-1,   # Neuron 0 (θ1): angle measurement
        1.0e-1,   # Neuron 1 (θ2): angle measurement
        1.0e-2,   # Neuron 2 (ω1): velocity measurement (noisier)
        1.0e-2    # Neuron 3 (ω2): velocity measurement (noisier)
    ]
    
    print("\n" + "="*70)
    print("PF-RHONN Neuron-Specific Parameters:")
    print("="*70)
    state_names = ['θ1 (angle 1)', 'θ2 (angle 2)', 'ω1 (ang vel 1)', 'ω2 (ang vel 2)']
    for i in range(n_states):
        print(f"  Neuron {i} ({state_names[i]}): Q_std={pf_Q_std[i]:.4f}, R_std={pf_R_std[i]:.4f}")
    
    # Initialize Trainers
    ekf = EKF_Trainer(n_states, eta=1.0, P0=1.0, Q=1e-3, R=1e-5)
    ukf = UKF_Trainer(n_states, eta=1.2, alpha=1e-2, Q_std=1e-2, R_std=1e-6)
    pf = PF_Trainer(n_states, n_particles=1100, Q_std=pf_Q_std, R_std=pf_R_std)
    
    # Set initial weights for all filters
    for i in range(n_states):
        ekf.weights[i] = initial_weights[i].copy()
        ukf.weights[i] = initial_weights[i].copy()
        # For particle filter, initialize all particles with the same weights
        pf.particles[i] = np.tile(initial_weights[i], (pf.n_particles, 1))
    
    print("\n✓ All filters initialized with the same RHONN weights")
    
    # Arrays for states
    x_true = np.zeros((n_steps, 4))
    x_est_ekf = np.zeros((n_steps, 4))
    x_est_ukf = np.zeros((n_steps, 4))
    x_est_pf = np.zeros((n_steps, 4))
    
    # Arrays for noisy measurements
    y_measured = np.zeros((n_steps, 4))
    
    # Variables para medir tiempos de entrenamiento
    ekf_training_times = []
    ukf_training_times = []
    pf_training_times = []
    
    # Initial Conditions (double pendulum hanging down with small perturbation)
    x_true[0] = [np.pi/6, np.pi/4, 0.0, 0.0]  # [θ1, θ2, ω1, ω2]
    x_est_ekf[0] = x_true[0]
    x_est_ukf[0] = x_true[0]
    x_est_pf[0] = x_true[0]
    
    # Add Gaussian noise to initial measurement
    initial_noise = np.array([
        np.random.normal(0, angle_noise_std),
        np.random.normal(0, angle_noise_std),
        np.random.normal(0, omega_noise_std),
        np.random.normal(0, omega_noise_std)
    ])
    y_measured[0] = x_true[0] + initial_noise
    
    # Excitation Input (external torques)
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        # Rich excitation signal with multiple frequencies
        τ1 = 0.5 * np.sin(1.0 * t[k]) + 0.3 * np.sin(2.5 * t[k])
        τ2 = 0.4 * np.sin(1.5 * t[k]) + 0.2 * np.cos(3.0 * t[k])
        
        # Add some step changes for better excitation
        if 400 < k < 450:
            τ1 += 1.0
            τ2 += 0.5
        elif 900 < k < 950:
            τ1 -= 0.8
            τ2 += 0.6
        elif 1200 < k < 1250:
            τ1 += 0.6
            τ2 -= 0.7
            
        u_hist[k] = [τ1, τ2]

    print("\n" + "="*70)
    print("Simulating Double Pendulum with SERIES CONFIGURATION (PF only)...")
    print("="*70)
    print("\nEstructura de características por neurona:")
    for i in range(n_states):
        print(f"  Neurona {i} ({state_names[i]}): {get_z_size(i)} características")
    
    print(f"\nConfiguration: PARALLEL (all filters use their own estimates)")
    print(f"Integration Method: Runge-Kutta 4th Order (RK4)")
    print(f"\nGaussian Noise Parameters:")
    print(f"  Process noise std: θ={process_noise_std[0]:.4f}, ω={process_noise_std[2]:.4f} rad/s")
    print(f"  Measurement noise std:")
    print(f"    - Angles (θ1, θ2): ±{angle_noise_std:.4f} rad")
    print(f"    - Angular velocities (ω1, ω2): ±{omega_noise_std:.4f} rad/s")
    print(f"\nIntegration Method: Runge-Kutta 4th Order (RK4)")
    print(f"Time step (dt): {dt} s")
    print(f"Simulation duration: {n_steps*dt:.1f} s ({n_steps} steps)")
    
    # Main simulation loop with PARALLEL CONFIGURATION
    for k in range(n_steps - 1):
        # 1. Generate true next state with process noise
        x_next_clean = plant(x_true[k], u_hist[k], dt)
        
        # Add Gaussian process noise
        process_noise = np.array([
            np.random.normal(0, process_noise_std[0]),
            np.random.normal(0, process_noise_std[1]),
            np.random.normal(0, process_noise_std[2]),
            np.random.normal(0, process_noise_std[3])
        ])
        x_true[k+1] = x_next_clean + process_noise
        
        # 2. Create noisy measurement with Gaussian noise
        measurement_noise = np.array([
            np.random.normal(0, angle_noise_std),
            np.random.normal(0, angle_noise_std),
            np.random.normal(0, omega_noise_std),
            np.random.normal(0, omega_noise_std)
        ])
        
        y_measured[k+1] = x_true[k+1] + measurement_noise
        
        # 3. Update filters with NOISY MEASUREMENTS
        # Each filter uses its OWN previous estimate as input
        
        # --- EKF Update & Predict ---
        start_time = time.perf_counter()
        ekf.update(y_measured[k+1], x_est_ekf[k], u_hist[k])
        ekf_time = time.perf_counter() - start_time
        ekf_training_times.append(ekf_time)
        
        # Predict next state using EKF's OWN estimate
        for i in range(4):
            z_ekf = construct_z_vector(y_measured[k], u_hist[k], i)
            x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z_ekf)
        
        # --- UKF Update & Predict ---
        start_time = time.perf_counter()
        ukf.update(y_measured[k+1], x_est_ukf[k], u_hist[k])
        ukf_time = time.perf_counter() - start_time
        ukf_training_times.append(ukf_time)
        
        # Predict next state using UKF's OWN estimate
        for i in range(4):
            z_ukf = construct_z_vector(y_measured[k], u_hist[k], i)
            x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z_ukf)
        
        # --- PF Update & Predict ---
        start_time = time.perf_counter()
        pf.update(x_true[k+1], x_true[k], u_hist[k])  # Using measurements instead of own estimates
        pf_time = time.perf_counter() - start_time
        pf_training_times.append(pf_time)
        
        w_pf = pf.get_estimates()
        # Predict next state using MEASUREMENTS (series configuration)
        for i in range(4):
            z_pf = construct_z_vector(y_measured[k], u_hist[k], i)  # Using measurements
            x_est_pf[k+1, i] = np.dot(w_pf[i], z_pf)
        
        # Progress reporting
        if k % 300 == 0 and k > 0:
            print(f"Step {k}/{n_steps-1}")
            # Show current errors
            err_ekf = np.linalg.norm(x_true[k] - x_est_ekf[k])
            err_ukf = np.linalg.norm(x_true[k] - x_est_ukf[k])
            err_pf = np.linalg.norm(x_true[k] - x_est_pf[k])
            print(f"  Current errors - EKF: {err_ekf:.4f}, UKF: {err_ukf:.4f}, PF: {err_pf:.4f}")

    # ============================================================
    # 6) Calculate Comprehensive Error Metrics
    # ============================================================
    
    print("\n" + "="*70)
    print("📊 COMPREHENSIVE RESULTS - PARALLEL CONFIGURATION WITH GAUSSIAN NOISE")
    print("="*70)
    # Calculate comprehensive error metrics for each filter
    metrics_ekf = calculate_error_metrics(x_true, x_est_ekf, "EKF-RHONN")
    metrics_ukf = calculate_error_metrics(x_true, x_est_ukf, "UKF-RHONN")
    metrics_pf = calculate_error_metrics(x_true, x_est_pf, "PF-RHONN")
    
    # Print comprehensive metrics
    print("\n--- EKF-RHONN Performance Metrics ---")
    print(f"RMSE total: {metrics_ekf['RMSE_total']:.6f}")
    for i, name in enumerate(state_names):
        print(f"  {name}: RMSE={metrics_ekf['RMSE'][i]:.6f}, MAE={metrics_ekf['MAE'][i]:.6f}, "
              f"NRMSE={metrics_ekf['NRMSE'][i]:.4f}, R²={metrics_ekf['R2'][i]:.4f}")
    
    print("\n--- UKF-RHONN Performance Metrics ---")
    print(f"RMSE total: {metrics_ukf['RMSE_total']:.6f}")
    for i, name in enumerate(state_names):
        print(f"  {name}: RMSE={metrics_ukf['RMSE'][i]:.6f}, MAE={metrics_ukf['MAE'][i]:.6f}, "
              f"NRMSE={metrics_ukf['NRMSE'][i]:.4f}, R²={metrics_ukf['R2'][i]:.4f}")
    
    print("\n--- PF-RHONN Performance Metrics ---")
    print(f"RMSE total: {metrics_pf['RMSE_total']:.6f}")
    for i, name in enumerate(state_names):
        print(f"  {name}: RMSE={metrics_pf['RMSE'][i]:.6f}, MAE={metrics_pf['MAE'][i]:.6f}, "
              f"NRMSE={metrics_pf['NRMSE'][i]:.4f}, R²={metrics_pf['R2'][i]:.4f}")
    
    # Análisis de tiempos de entrenamiento
    print("\n" + "="*70)
    print("⏱️  TIEMPOS DE ENTRENAMIENTO POR FILTRO")
    print("="*70)
    
    # Calcular estadísticas de tiempos
    ekf_total_time = np.sum(ekf_training_times)
    ukf_total_time = np.sum(ukf_training_times)
    pf_total_time = np.sum(pf_training_times)
    
    ekf_mean_time = np.mean(ekf_training_times)
    ukf_mean_time = np.mean(ukf_training_times)
    pf_mean_time = np.mean(pf_training_times)
    
    ekf_std_time = np.std(ekf_training_times)
    ukf_std_time = np.std(ukf_training_times)
    pf_std_time = np.std(pf_training_times)
    
    print(f"\n📈 Estadísticas de Tiempos por Iteración (en segundos):")
    print(f"\nEKF-RHONN:")
    print(f"  Total: {ekf_total_time:.6f} s | Media: {ekf_mean_time*1000:.4f} ms")
    print(f"  Std: {ekf_std_time*1000:.4f} ms | Min: {np.min(ekf_training_times)*1000:.4f} ms")
    print(f"  Max: {np.max(ekf_training_times)*1000:.4f} ms")
    
    print(f"\nUKF-RHONN:")
    print(f"  Total: {ukf_total_time:.6f} s | Media: {ukf_mean_time*1000:.4f} ms")
    print(f"  Std: {ukf_std_time*1000:.4f} ms | Min: {np.min(ukf_training_times)*1000:.4f} ms")
    print(f"  Max: {np.max(ukf_training_times)*1000:.4f} ms")
    
    print(f"\nPF-RHONN:")
    print(f"  Total: {pf_total_time:.6f} s | Media: {pf_mean_time*1000:.4f} ms")
    print(f"  Std: {pf_std_time*1000:.4f} ms | Min: {np.min(pf_training_times)*1000:.4f} ms")
    print(f"  Max: {np.max(pf_training_times)*1000:.4f} ms")
    
    # Comparación relativa
    print(f"\n⚡ Comparación Relativa de Velocidad:")
    print(f"  EKF es {ukf_mean_time/ekf_mean_time:.2f}x más rápido que UKF")
    print(f"  EKF es {pf_mean_time/ekf_mean_time:.2f}x más rápido que PF")
    print(f"  UKF es {pf_mean_time/ukf_mean_time:.2f}x más rápido que PF")
    
    # Calcular eficiencia (precisión por unidad de tiempo)
    print(f"\n🎯 Eficiencia (R² / Tiempo de Entrenamiento):")
    print(f"  EKF: {metrics_ekf['R2_total']/ekf_total_time:.4f} R²/s")
    print(f"  UKF: {metrics_ukf['R2_total']/ukf_total_time:.4f} R²/s")
    print(f"  PF:  {metrics_pf['R2_total']/pf_total_time:.4f} R²/s")
    
    # Analysis of noise characteristics
    print("\n" + "="*70)
    print("📉 ANÁLISIS DE RUIDO (Gaussian)")
    print("="*70)
    
    # Calculate measurement noise statistics
    meas_noise = y_measured - x_true
    print(f"\nCaracterísticas del ruido de medición:")
    for i, name in enumerate(state_names):
        noise_samples = meas_noise[:, i]
        print(f"\n  {name}:")
        print(f"    Std: {np.std(noise_samples):.6f}")
        print(f"    Mean: {np.mean(noise_samples):.6f} (should be ~0 for Gaussian)")
        print(f"    Max abs: {np.max(np.abs(noise_samples)):.6f}")
    
    print("\n" + "="*70)
    print("🏆 MEJOR FILTRO: ", end="")
    rmse_dict = {'EKF-RHONN': metrics_ekf['RMSE_total'], 
                 'UKF-RHONN': metrics_ukf['RMSE_total'], 
                 'PF-RHONN': metrics_pf['RMSE_total']}
    best_filter = min(rmse_dict, key=rmse_dict.get)
    print(f"{best_filter} (RMSE total: {rmse_dict[best_filter]:.6f})")
    print("="*70)

Initial RHONN weights (uniform distribution):
  Neuron 0: shape=(3,), min=-0.7101, max=0.3165, mean=-0.0699
  Neuron 1: shape=(4,), min=-0.6689, max=0.9859, mean=-0.2045
  Neuron 2: shape=(4,), min=-0.7763, max=0.7724, mean=-0.2666
  Neuron 3: shape=(4,), min=-0.8238, max=0.7480, mean=-0.0612

PF-RHONN Neuron-Specific Parameters:
  Neuron 0 (θ1 (angle 1)): Q_std=0.5000, R_std=0.1000
  Neuron 1 (θ2 (angle 2)): Q_std=0.5000, R_std=0.1000
  Neuron 2 (ω1 (ang vel 1)): Q_std=0.2000, R_std=0.0100
  Neuron 3 (ω2 (ang vel 2)): Q_std=0.2000, R_std=0.0100

✓ All filters initialized with the same RHONN weights

Simulating Double Pendulum with SERIES CONFIGURATION (PF only)...

Estructura de características por neurona:
  Neurona 0 (θ1 (angle 1)): 3 características
  Neurona 1 (θ2 (angle 2)): 4 características
  Neurona 2 (ω1 (ang vel 1)): 4 características
  Neurona 3 (ω2 (ang vel 2)): 4 características

Configuration: PARALLEL (all filters use their own estimates)
Integration Method: Runge-Kutta

In [ ]:
# ============================================================
# 7) Visualization - Double Pendulum
# ============================================================

print("\nGenerando visualizaciones...")

# Configuración de formato para tesis
thesis_config = {
    'font_family': 'Computer Modern, serif',
    'font_size': 16,
    'title_font_size': 18,
    'legend_font_size': 14,
    'line_width_true': 2.5,
    'line_width_est': 2.0,
    'line_width_meas': 1.0,
    'plot_width': 1000,
    'plot_height': 500,
    'grid_color': 'rgba(200, 200, 200, 0.3)',
    'grid_width': 0.5
}

# --- Gráfica de tiempos de entrenamiento ---
fig_times = go.Figure()

# Crear arrays de tiempo de simulación para el eje x
sim_time = t[1:]  # Excluir el primer tiempo (k=0)

# Agregar trazas de tiempo para cada filtro
# fig_times.add_trace(go.Scatter(
#     x=sim_time, y=ekf_training_times,
#     mode='lines',
#     name='EKF-RHONN',
#     line=dict(color='#1f77b4', width=2),
# ))
# 
# fig_times.add_trace(go.Scatter(
#     x=sim_time, y=ukf_training_times,
#     mode='lines',
#     name='UKF-RHONN',
#     line=dict(color='#2ca02c', width=2),
# ))

fig_times.add_trace(go.Scatter(
    x=sim_time, y=pf_training_times,
    mode='lines',
    name='PF-RHONN',
    line=dict(color='#d62728', width=2),
))

fig_times.update_layout(
    title={
        'text': 'Tiempos de Entrenamiento por Iteración',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo de Simulación (s)',
    yaxis_title='Tiempo de Entrenamiento (s)',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        type='log',  # Escala logarítmica para mejor visualización
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_times.show()

# --- Gráfica de barras comparando tiempos medios ---
fig_times_bar = go.Figure()

filters = ['EKF-RHONN', 'UKF-RHONN', 'PF-RHONN']
mean_times_ms = [ekf_mean_time*1000, ukf_mean_time*1000, pf_mean_time*1000]

fig_times_bar.add_trace(go.Bar(
    x=filters,
    y=mean_times_ms,
    marker_color=['#1f77b4', '#2ca02c', '#d62728'],
    text=[f'{t:.3f} ms' for t in mean_times_ms],
    textposition='outside'
))

fig_times_bar.update_layout(
    title={
        'text': 'Tiempo Promedio de Entrenamiento por Iteración',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tipo de Filtro',
    yaxis_title='Tiempo Promedio (ms)',
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    xaxis=dict(
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_times_bar.show()

# --- Visualización de la trayectoria del doble péndulo (animación conceptual) ---
# Calcular posiciones cartesianas de las masas del péndulo
l1, l2 = 1.0, 1.2  # longitudes de los péndulos

x1_true = l1 * np.sin(x_true[:, 0])
y1_true = -l1 * np.cos(x_true[:, 0])
x2_true = x1_true + l2 * np.sin(x_true[:, 1])
y2_true = y1_true - l2 * np.cos(x_true[:, 1])

# Trayectoria de la segunda masa (la más interesante)
fig_traj = go.Figure()

fig_traj.add_trace(go.Scatter(
    x=x2_true, y=y2_true,
    mode='lines',
    name='Trayectoria Real (Masa 2)',
    line=dict(color='#000000', width=2),
))

# Puntos de inicio y final
fig_traj.add_trace(go.Scatter(
    x=[x2_true[0]], y=[y2_true[0]],
    mode='markers',
    name='Inicio',
    marker=dict(size=12, color='green', symbol='circle'),
))

fig_traj.add_trace(go.Scatter(
    x=[x2_true[-1]], y=[y2_true[-1]],
    mode='markers',
    name='Final',
    marker=dict(size=12, color='red', symbol='square'),
))

fig_traj.update_layout(
    title={
        'text': 'Trayectoria del Doble Péndulo (Masa 2)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Posición x (m)',
    yaxis_title='Posición y (m)',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5,
    ),
    yaxis=dict(
        scaleanchor="x",
        scaleratio=1,
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(255, 255, 255, 0)',
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_width'],  # Cuadrado
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_traj.show()

# --- Gráficas por Estado (con medidas ruidosas) ---
states_info = [
    {'idx': 0, 'var': 'θ₁', 'desc': 'Ángulo del Primer Péndulo', 'y_label': 'Ángulo θ₁ (rad)'},
    {'idx': 1, 'var': 'θ₂', 'desc': 'Ángulo del Segundo Péndulo', 'y_label': 'Ángulo θ₂ (rad)'},
    {'idx': 2, 'var': 'ω₁', 'desc': 'Velocidad Angular 1', 'y_label': 'Velocidad ω₁ (rad/s)'},
    {'idx': 3, 'var': 'ω₂', 'desc': 'Velocidad Angular 2', 'y_label': 'Velocidad ω₂ (rad/s)'}
]

for state_info in states_info:
    i = state_info['idx']
    
    fig = go.Figure()
    
    # Measured state (with noise)
    fig.add_trace(go.Scatter(
        x=t, y=y_measured[:, i],
        mode='lines',
        name='Medición (con ruido)',
        line=dict(color='#808080', width=thesis_config['line_width_true']),
        showlegend=True
    ))
    
    # EKF estimate
    fig.add_trace(go.Scatter(
        x=t, y=x_est_ekf[:, i],
        mode='lines',
        name='EKF-RHONN',
        line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
        showlegend=True
    ))
    
    # UKF estimate
    fig.add_trace(go.Scatter(
        x=t, y=x_est_ukf[:, i],
        mode='lines',
        name='UKF-RHONN',
        line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
        showlegend=True
    ))
    
    # PF estimate
    fig.add_trace(go.Scatter(
        x=t, y=x_est_pf[:, i],
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
        showlegend=True
    ))
    
    fig.update_layout(
        title={
            'text': f'Estado {state_info["var"]}: {state_info["desc"]}',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title=state_info['y_label'],
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='center',
            x=0.5,
            bgcolor='rgba(255, 255, 255, 0)',
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig.show()

# --- Gráfica de errores acumulados ---
fig_errors = go.Figure()

# Calculate cumulative errors over time
cumulative_error_ekf = np.zeros(n_steps)
cumulative_error_ukf = np.zeros(n_steps)
cumulative_error_pf = np.zeros(n_steps)

for k in range(1, n_steps):
    error_ekf = np.linalg.norm(x_true[k] - x_est_ekf[k])
    error_ukf = np.linalg.norm(x_true[k] - x_est_ukf[k])
    error_pf = np.linalg.norm(x_true[k] - x_est_pf[k])
    
    cumulative_error_ekf[k] = cumulative_error_ekf[k-1] + error_ekf
    cumulative_error_ukf[k] = cumulative_error_ukf[k-1] + error_ukf
    cumulative_error_pf[k] = cumulative_error_pf[k-1] + error_pf

fig_errors.add_trace(go.Scatter(
    x=t, y=cumulative_error_ekf,
    mode='lines',
    name='EKF-RHONN',
    line=dict(color='#1f77b4', width=2),
))

fig_errors.add_trace(go.Scatter(
    x=t, y=cumulative_error_ukf,
    mode='lines',
    name='UKF-RHONN',
    line=dict(color='#2ca02c', width=2),
))

fig_errors.add_trace(go.Scatter(
    x=t, y=cumulative_error_pf,
    mode='lines',
    name='PF-RHONN',
    line=dict(color='#d62728', width=2),
))

fig_errors.update_layout(
    title={
        'text': 'Error Acumulado en el Tiempo - Norma Euclidiana',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo (s)',
    yaxis_title='Error Acumulado',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(255, 255, 255, 0)',
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_errors.show()

# --- Gráfica de compensación tiempo-precision (Pareto) ---
fig_pareto = go.Figure()

fig_pareto.add_trace(go.Scatter(
    x=[ekf_mean_time*1000, ukf_mean_time*1000, pf_mean_time*1000],
    y=[metrics_ekf['RMSE_total'], metrics_ukf['RMSE_total'], metrics_pf['RMSE_total']],
    mode='markers+text',
    text=['EKF', 'UKF', 'PF'],
    textposition='top center',
    marker=dict(
        size=15,
        color=['#1f77b4', '#2ca02c', '#d62728'],
        line=dict(width=2, color='DarkSlateGrey')
    ),
    name='Filtros'
))

fig_pareto.update_layout(
    title={
        'text': 'Compensación Tiempo-Precisión (Frente de Pareto)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo Promedio por Iteración (ms)',
    yaxis_title='Error Cuadrático Medio (RMSE)',
    xaxis=dict(
        type='log',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        type='log',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_pareto.show()

# --- Gráfica de comparación de métricas ---
fig_metrics = go.Figure()

metrics_names = ['RMSE', 'MAE', 'NRMSE']
ekf_vals = [metrics_ekf['RMSE_total'], metrics_ekf['MAE_total'], metrics_ekf['NRMSE_total']]
ukf_vals = [metrics_ukf['RMSE_total'], metrics_ukf['MAE_total'], metrics_ukf['NRMSE_total']]
pf_vals = [metrics_pf['RMSE_total'], metrics_pf['MAE_total'], metrics_pf['NRMSE_total']]

x = np.arange(len(metrics_names))
width = 0.25

fig_metrics.add_trace(go.Bar(
    x=x - width, y=ekf_vals, width=width,
    name='EKF-RHONN',
    marker_color='#1f77b4'
))

fig_metrics.add_trace(go.Bar(
    x=x, y=ukf_vals, width=width,
    name='UKF-RHONN',
    marker_color='#2ca02c'
))

fig_metrics.add_trace(go.Bar(
    x=x + width, y=pf_vals, width=width,
    name='PF-RHONN',
    marker_color='#000080'
))

fig_metrics.update_layout(
    title={
        'text': 'Métricas de Error',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis=dict(
        tickmode='array',
        tickvals=x,
        ticktext=metrics_names,
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        title='Valor de la Métrica',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
                orientation='h',
                yanchor='bottom',
                y=1.02,
                xanchor='center',
                x=0.5,
                bgcolor='rgba(255, 255, 255, 0)',
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    # legend=dict(
    #     x=0.02,
    #     y=0.98,
    #     xanchor='left',
    #     yanchor='top',
    #     bgcolor='rgba(255, 255, 255, 0.9)',
    #     bordercolor='black',
    #     borderwidth=1,
    #     font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    # ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=100, b=60),
    barmode='group'
)

fig_metrics.show()

print("\n✅ Visualización completa con configuración paralela y análisis de métricas.")

# ============================================================
# 8) Additional Analysis
# ============================================================

print("\n" + "="*70)
print("📈 ANÁLISIS ADICIONAL")
print("="*70)

# Analyze convergence
convergence_window = 100  # Look at last 100 steps
if n_steps > convergence_window:
    final_metrics = {
        # 'EKF': calculate_error_metrics(x_true[-convergence_window:, :], 
        #                                x_est_ekf[-convergence_window:, :]),
        # 'UKF': calculate_error_metrics(x_true[-convergence_window:, :], 
        #                                x_est_ukf[-convergence_window:, :]),
        'PF': calculate_error_metrics(x_true[-convergence_window:, :], 
                                      x_est_pf[-convergence_window:, :])
    }
    
    print(f"\nMétricas en los últimos {convergence_window} pasos (estado estacionario):")
    for filter_name, metrics in final_metrics.items():
        print(f"\n{filter_name}:")
        print(f"  RMSE total: {metrics['RMSE_total']:.6f}")
        for i, name in enumerate(state_names):
            print(f"    {name}: RMSE={metrics['RMSE'][i]:.6f}, R²={metrics['R2'][i]:.4f}")

# Resumen final
print("\n" + "="*70)
print("🎯 RESUMEN EJECUTIVO - PF-RHONN EVALUATION")
print("="*70)
print(f"PF-RHONN RMSE total: {metrics_pf['RMSE_total']:.6f}")
print(f"PF-RHONN R² total: {metrics_pf['R2_total']:.4f}")
print(f"PF-RHONN Mean training time: {pf_mean_time*1000:.4f} ms")
print("\nIntegration Method Used: Runge-Kutta 4th Order (RK4) - High accuracy discretization")
print("Error Metrics: RMSE, MAE, NRMSE, R², Max Error")



Generando visualizaciones...



✅ Visualización completa con configuración paralela y análisis de métricas.

📈 ANÁLISIS ADICIONAL

Métricas en los últimos 100 pasos (estado estacionario):

PF:
  RMSE total: 0.009699
    θ1 (angle 1): RMSE=0.009486, R²=0.9968
    θ2 (angle 2): RMSE=0.007292, R²=0.9978
    ω1 (ang vel 1): RMSE=0.008812, R²=0.9999
    ω2 (ang vel 2): RMSE=0.012468, R²=0.9978

🎯 RESUMEN EJECUTIVO - PF-RHONN EVALUATION
PF-RHONN RMSE total: 0.014494
PF-RHONN R² total: 0.9991
PF-RHONN Mean training time: 0.6234 ms

Integration Method Used: Runge-Kutta 4th Order (RK4) - High accuracy discretization
Error Metrics: RMSE, MAE, NRMSE, R², Max Error


In [ ]:
# Imprimir pesos finales de cada filtro
print("\n" + "="*70)
print("🔍 PESOS FINALES DE LAS REDES NEURONALES")
print("="*70)

print("\n--- EKF-RHONN ---")
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {ekf.weights[i]}")
    print(f"  Norm: {np.linalg.norm(ekf.weights[i]):.4f}, Mean: {np.mean(ekf.weights[i]):.4f}, Std: {np.std(ekf.weights[i]):.4f}")

print("\n--- UKF-RHONN ---")
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {ukf.weights[i]}")
    print(f"  Norm: {np.linalg.norm(ukf.weights[i]):.4f}, Mean: {np.mean(ukf.weights[i]):.4f}, Std: {np.std(ukf.weights[i]):.4f}")

print("\n--- PF-RHONN ---")
w_pf_final = pf.get_estimates()
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {w_pf_final[i]}")
    print(f"  Norm: {np.linalg.norm(w_pf_final[i]):.4f}, Mean: {np.mean(w_pf_final[i]):.4f}, Std: {np.std(w_pf_final[i]):.4f}")

print("\n" + "="*70)
print("✅ SIMULACIÓN COMPLETADA - DOBLE PÉNDULO")
print("="*70)
print("\nSistema: Doble Péndulo con Amortiguamiento")
print(f"Método de discretización: Runge-Kutta 4° Orden (RK4)")
print(f"Paso de tiempo: {dt} s")
print(f"Duración: {n_steps*dt:.1f} s")
print(f"Número de estados: {n_states}")
print("\nMétricas de error utilizadas:")
print("  - RMSE (Root Mean Square Error)")
print("  - MAE (Mean Absolute Error)")
print("  - NRMSE (Normalized Root Mean Square Error)")
print("  - R² (Coefficient of Determination)")
print("  - Max Error (Maximum Absolute Error)")



🔍 PESOS FINALES DE LAS REDES NEURONALES

--- EKF-RHONN ---

Neurona 0 (θ1 (angle 1)) - 3 pesos:
  [ 1.61605216 -0.64138434 -1.88645283]
  Norm: 2.5655, Mean: -0.3039, Std: 1.4497

Neurona 1 (θ2 (angle 2)) - 4 pesos:
  [-0.66457151  0.88441442 -1.0767424  -0.32526774]
  Norm: 1.5777, Mean: -0.2955, Std: 0.7314

Neurona 2 (ω1 (ang vel 1)) - 4 pesos:
  [ 1.71690411  0.89602711 -1.03513717 -0.4837075 ]
  Norm: 2.2486, Mean: 0.2735, Std: 1.0905

Neurona 3 (ω2 (ang vel 2)) - 4 pesos:
  [ 1.16768338  1.10022974 -0.93115745 -0.4460336 ]
  Norm: 1.9079, Mean: 0.2227, Std: 0.9276

--- UKF-RHONN ---

Neurona 0 (θ1 (angle 1)) - 3 pesos:
  [ 2.29706453  0.17757839 -3.28786689]
  Norm: 4.0147, Mean: -0.2711, Std: 2.3020

Neurona 1 (θ2 (angle 2)) - 4 pesos:
  [-2.72965412  0.86741953  4.02121698 -0.15346362]
  Norm: 4.9393, Mean: 0.5014, Std: 2.4182

Neurona 2 (ω1 (ang vel 1)) - 4 pesos:
  [ 0.72361495  3.01809448 -2.00855914 -0.07590985]
  Norm: 3.6976, Mean: 0.4143, Std: 1.8018

Neurona 3 (ω2 (ang